# A3 cross-session re-ID — BCI Competition IV-2a

Realistic biometric-linkage scenario: 9 subjects recorded across two sessions on different days. Probe trained on session-1 embeddings, tested on session-2 embeddings. PhysioNet has only one session per subject so cannot answer this directly — IV-2a is the public testbed.

Send back `results/05_a3_cross_session.json` and `runs/<run_id>/meta.json`.

**Runtime → L4 GPU.** Expected wall ~10 min (small dataset).

**Don't `Save a copy in GitHub` from Colab.**

## 1. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install project deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Run A3 cross-session re-ID — all three victim families

moabb downloads BCI IV-2a (~50 MB) on first call. No separate prefetcher needed.

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.05_a3_cross_session --all

## 5. Emit run metadata + result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch

PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

EXPERIMENT = 'experiments.05_a3_cross_session'
ARGS = ['--all']
RESULTS = 'results/05_a3_cross_session.json'
FIGURE = 'figures/05_a3_cross_session.pdf'
TAG = 'a3_cross_session'

try:
    git_sha = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR
    ).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"')
    git_sha = 'unknown'

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'

meta = {
    'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS,
    'git_sha': git_sha, 'hardware': f'Colab {gpu}',
    'platform': platform.platform(), 'torch_version': torch.__version__,
    'completed_at_utc': now_utc, 'outputs': [RESULTS, FIGURE],
}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('=== run metadata ===')
print(json.dumps(meta, indent=2))
print()

if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing — Cell 4 did not finish. Re-run Cell 4, then this cell.')
print(f'=== {RESULTS} ===')
print(open(RESULTS).read())

## 6. Download artifacts

In [ ]:
from google.colab import files
files.download('results/05_a3_cross_session.json')
files.download(f'runs/{run_id}/meta.json')